# Project Goal

Build an end-to-end PySpark medallion pipeline for flight delay analytics.  
The project focuses on production-oriented engineering practices rather than exploratory data analysis.

Data pipeline are organized in Medallion Architecture design pattern, containing layers:
- **Bronze** → raw ingestion + basic validation
- **Silver** → cleaned, standardized, enriched data
- **Gold** → star schema for analytics and ML
- **Data Marts** → a subset of a data warehouse designed to serve the needs of a specific department or business unit

## DWH structure:
- dimensions:
    - dim_time - created only once at the beginning of the project
    - dim_airports - SCD2
    - dim_airlines - SCD2
- facts:
    - fct_flights

#### SCD2 Logic
1. Load current dimension
2. Load incoming batch
3. Detect changed records using row hash
4. Expire old versions
5. Insert new versions
6. Rebuild dimension table

# Notebook tags description
Tasks have 2 labels:
- "PIPELINE" - this part finally should go to data workflow
- "ANALYSIS" - this part is for some additional, often necessary check and findings, but this part should not be in data workflow

In [1]:
import os
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession, SQLContext
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType
)

import warnings

warnings.filterwarnings("ignore")

# Environment variables

In [2]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/organizations/usdot/flight-delays/airports.csv
/kaggle/input/datasets/organizations/usdot/flight-delays/airlines.csv
/kaggle/input/datasets/organizations/usdot/flight-delays/flights.csv


In [3]:
ENV_file_name_fligths = 'flights.csv'
ENV_file_name_airports = 'airports.csv'
ENV_file_name_airlines = 'airlines.csv'

ENV_raw_data_path = '/kaggle/input/datasets/organizations/usdot/flight-delays'


In [4]:
spark = SparkSession.builder.appName("Flight_DWH").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/09 10:09:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Bronze
Bronze layer stores raw data with minimal transformations.

Objectives:
- load source data
- standardize schema
- add technical metadata
- perform initial validation

## BRONZE - PIPELINE - Load data
Load 3 files into 3 spark dataframes df_flights, df_airports and df_airlines

### Schema Definition

### Flights Reference Dataset Schema

| Column | Data Type | Notes / Comments |
|---|---|---|
| YEAR | IntegerType | Flight year |
| MONTH | IntegerType | Month of flight (1–12) |
| DAY | IntegerType | Day of month (1–31) |
| DAY_OF_WEEK | IntegerType | Day of week (1–7) |
| AIRLINE | StringType | Airline carrier code |
| FLIGHT_NUMBER | IntegerType | Airline-specific flight number |
| TAIL_NUMBER | StringType | Aircraft registration identifier |
| ORIGIN_AIRPORT | StringType | Departure airport code |
| DESTINATION_AIRPORT | StringType | Arrival airport code |
| SCHEDULED_DEPARTURE | IntegerType | Planned departure time in HHMM format |
| DEPARTURE_TIME | IntegerType | Actual departure time in HHMM format |
| DEPARTURE_DELAY | IntegerType | Departure delay in minutes (negative = early) |
| TAXI_OUT | IntegerType | Minutes from gate departure to takeoff |
| WHEELS_OFF | IntegerType | Actual takeoff time in HHMM format |
| SCHEDULED_TIME | IntegerType | Planned total flight duration in minutes |
| ELAPSED_TIME | IntegerType | Actual total flight duration in minutes |
| AIR_TIME | IntegerType | Time spent airborne in minutes |
| DISTANCE | IntegerType | Flight distance in miles |
| WHEELS_ON | IntegerType | Landing time in HHMM format |
| TAXI_IN | IntegerType | Minutes from landing to gate arrival |
| SCHEDULED_ARRIVAL | IntegerType | Planned arrival time in HHMM format |
| ARRIVAL_TIME | IntegerType | Actual arrival time in HHMM format |
| ARRIVAL_DELAY | IntegerType | Arrival delay in minutes (negative = early) |
| DIVERTED | IntegerType | Binary flag: 1 = diverted flight |
| CANCELLED | IntegerType | Binary flag: 1 = cancelled flight |
| CANCELLATION_REASON | StringType | Cancellation reason code (if cancelled) |
| AIR_SYSTEM_DELAY | IntegerType | Delay caused by air traffic / airport operations |
| SECURITY_DELAY | IntegerType | Delay caused by security issues |
| AIRLINE_DELAY | IntegerType | Delay attributed to airline operations |
| LATE_AIRCRAFT_DELAY | IntegerType | Delay caused by late incoming aircraft |
| WEATHER_DELAY | IntegerType | Delay caused by weather conditions |

In [5]:
schema_flights = StructType([
    StructField("YEAR", IntegerType(), True, {"comment": "Flight year"}),
    StructField("MONTH", IntegerType(), True, {"comment": "Month (1–12)"}),
    StructField("DAY", IntegerType(), True, {"comment": "Day of month"}),
    StructField("DAY_OF_WEEK", IntegerType(), True, {"comment": "Day of week (1–7)"}),

    StructField("AIRLINE", StringType(), True, {"comment": "Airline carrier code"}),
    StructField("FLIGHT_NUMBER", IntegerType(), True, {"comment": "Flight identifier"}),
    StructField("TAIL_NUMBER", StringType(), True, {"comment": "Aircraft registration"}),

    StructField("ORIGIN_AIRPORT", StringType(), True, {"comment": "Departure airport code"}),
    StructField("DESTINATION_AIRPORT", StringType(), True, {"comment": "Arrival airport code"}),

    StructField("SCHEDULED_DEPARTURE", IntegerType(), True, {"comment": "Scheduled departure (HHMM)"}),
    StructField("DEPARTURE_TIME", IntegerType(), True, {"comment": "Actual departure (HHMM)"}),
    StructField("DEPARTURE_DELAY", IntegerType(), True, {"comment": "Delay in minutes"}),

    StructField("TAXI_OUT", IntegerType(), True, {"comment": "Taxi-out time (min)"}),
    StructField("WHEELS_OFF", IntegerType(), True, {"comment": "Takeoff time (HHMM)"}),

    StructField("SCHEDULED_TIME", IntegerType(), True, {"comment": "Planned duration (min)"}),
    StructField("ELAPSED_TIME", IntegerType(), True, {"comment": "Actual duration (min)"}),
    StructField("AIR_TIME", IntegerType(), True, {"comment": "Air time (min)"}),
    StructField("DISTANCE", IntegerType(), True, {"comment": "Distance (miles)"}),

    StructField("WHEELS_ON", IntegerType(), True, {"comment": "Landing time (HHMM)"}),
    StructField("TAXI_IN", IntegerType(), True, {"comment": "Taxi-in time (min)"}),

    StructField("SCHEDULED_ARRIVAL", IntegerType(), True, {"comment": "Scheduled arrival (HHMM)"}),
    StructField("ARRIVAL_TIME", IntegerType(), True, {"comment": "Actual arrival (HHMM)"}),
    StructField("ARRIVAL_DELAY", IntegerType(), True, {"comment": "Arrival delay in minutes"}),

    StructField("DIVERTED", IntegerType(), True, {"comment": "1 = diverted flight"}),
    StructField("CANCELLED", IntegerType(), True, {"comment": "1 = cancelled flight"}),
    StructField("CANCELLATION_REASON", StringType(), True, {"comment": "Cancellation reason code"}),

    StructField("AIR_SYSTEM_DELAY", IntegerType(), True, {"comment": "Air system delay"}),
    StructField("SECURITY_DELAY", IntegerType(), True, {"comment": "Security delay"}),
    StructField("AIRLINE_DELAY", IntegerType(), True, {"comment": "Airline delay"}),
    StructField("LATE_AIRCRAFT_DELAY", IntegerType(), True, {"comment": "Late aircraft delay"}),
    StructField("WEATHER_DELAY", IntegerType(), True, {"comment": "Weather delay"})
])

### Airport Reference Dataset Schema

| Column | Data Type | Notes / Comments |
|---|---|---|
| IATA_CODE | StringType | Unique airport IATA code (3-letter identifier) |
| AIRPORT | StringType | Full airport name |
| CITY | StringType | City where airport is located |
| STATE | StringType | State / region (mainly US-based dataset) |
| COUNTRY | StringType | Country of airport |
| LATITUDE | DoubleType | Geographic latitude coordinate |
| LONGITUDE | DoubleType | Geographic longitude coordinate |

In [6]:
schema_airports = StructType([
    StructField("IATA_CODE", StringType(), True, {"comment": "Unique airport IATA code (3-letter)"}),
    StructField("AIRPORT", StringType(), True, {"comment": "Full airport name"}),
    StructField("CITY", StringType(), True, {"comment": "City of airport location"}),
    StructField("STATE", StringType(), True, {"comment": "State or region"}),
    StructField("COUNTRY", StringType(), True, {"comment": "Country"}),
    StructField("LATITUDE", DoubleType(), True, {"comment": "Latitude coordinate"}),
    StructField("LONGITUDE", DoubleType(), True, {"comment": "Longitude coordinate"})
])

### Airline Reference Dataset Schema

| Column | Data Type | Notes / Comments |
|---|---|---|
| IATA_CODE | StringType | Unique airline IATA code (2-letter identifier) |
| AIRLINE | StringType | Full airline name |

In [7]:
schema_airlines = StructType([
    StructField("IATA_CODE", StringType(), True, {"comment": "Airline IATA code (2-letter)"}),
    StructField("AIRLINE", StringType(), True, {"comment": "Full airline name"})
])

In [8]:
df_flights_bronze = spark.read.csv(os.path.join(ENV_raw_data_path, ENV_file_name_fligths), header=True, inferSchema=True)
df_airports_bronze = spark.read.csv(os.path.join(ENV_raw_data_path, ENV_file_name_airports), header=True, inferSchema=True)
df_airlines_bronze = spark.read.csv(os.path.join(ENV_raw_data_path, ENV_file_name_airlines), header=True, inferSchema=True)

## BRONZE - ANALYSIS - Raw data inspection
- check sample data (eg. 5 rows)
- check columns names (whitespaces, upper/lower/camel case)

In [9]:
display(df_flights_bronze.limit(30).toPandas())
df_flights_bronze.columns

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408,-22,0,0,None,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741,-9,0,0,None,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811,5,0,0,None,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756,-9,0,0,None,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259,-21,0,0,None,NaN,NaN,NaN,NaN,NaN
5,2015,1,1,4,DL,806,N3730B,SFO,MSP,25,...,610,8,0,0,None,NaN,NaN,NaN,NaN,NaN
6,2015,1,1,4,NK,612,N635NK,LAS,MSP,25,...,509,-17,0,0,None,NaN,NaN,NaN,NaN,NaN
7,2015,1,1,4,US,2013,N584UW,LAX,CLT,30,...,753,-10,0,0,None,NaN,NaN,NaN,NaN,NaN
8,2015,1,1,4,AA,1112,N3LAAA,SFO,DFW,30,...,532,-13,0,0,None,NaN,NaN,NaN,NaN,NaN
9,2015,1,1,4,DL,1173,N826DN,LAS,ATL,30,...,656,-15,0,0,None,NaN,NaN,NaN,NaN,NaN


['YEAR',
 'MONTH',
 'DAY',
 'DAY_OF_WEEK',
 'AIRLINE',
 'FLIGHT_NUMBER',
 'TAIL_NUMBER',
 'ORIGIN_AIRPORT',
 'DESTINATION_AIRPORT',
 'SCHEDULED_DEPARTURE',
 'DEPARTURE_TIME',
 'DEPARTURE_DELAY',
 'TAXI_OUT',
 'WHEELS_OFF',
 'SCHEDULED_TIME',
 'ELAPSED_TIME',
 'AIR_TIME',
 'DISTANCE',
 'WHEELS_ON',
 'TAXI_IN',
 'SCHEDULED_ARRIVAL',
 'ARRIVAL_TIME',
 'ARRIVAL_DELAY',
 'DIVERTED',
 'CANCELLED',
 'CANCELLATION_REASON',
 'AIR_SYSTEM_DELAY',
 'SECURITY_DELAY',
 'AIRLINE_DELAY',
 'LATE_AIRCRAFT_DELAY',
 'WEATHER_DELAY']

In [10]:
display(df_airports_bronze.limit(30).toPandas())
df_airports_bronze.columns

,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919
3,ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183
4,ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447
5,ACK,Nantucket Memorial Airport,Nantucket,MA,USA,41.25305,-70.06018
6,ACT,Waco Regional Airport,Waco,TX,USA,31.61129,-97.23052
7,ACV,Arcata Airport,Arcata/Eureka,CA,USA,40.97812,-124.10862
8,ACY,Atlantic City International Airport,Atlantic City,NJ,USA,39.45758,-74.57717
9,ADK,Adak Airport,Adak,AK,USA,51.87796,-176.64603


['IATA_CODE', 'AIRPORT', 'CITY', 'STATE', 'COUNTRY', 'LATITUDE', 'LONGITUDE']

In [11]:
display(df_airlines_bronze.limit(30).toPandas())
df_airlines_bronze.columns

,IATA_CODE,AIRLINE
0,UA,United Air Lines Inc.
1,AA,American Airlines Inc.
2,US,US Airways Inc.
3,F9,Frontier Airlines Inc.
4,B6,JetBlue Airways
5,OO,Skywest Airlines Inc.
6,AS,Alaska Airlines Inc.
7,NK,Spirit Air Lines
8,WN,Southwest Airlines Co.
9,DL,Delta Air Lines Inc.


['IATA_CODE', 'AIRLINE']

## BRONZE - ANALYSIS - Cleaning recommendations
Based on above quick look for above data:
- change iata_code for more informative column name
- change airlline into arline_code in flights
- create flights_date from year, month and day
- create timestamps base on:
    - SCHEDULED_DEPARTURE
    - DEPARTURE_TIME
    - WHEELS_OFF
    - WHEELS_ON
    - SCHEDULED_ARRIVAL
    - ARRIVAL_TIME

## BRONZE - ANALYSIS - Check number of records

In [12]:
print(f'Number of flights: {df_flights_bronze.count()}')
print(f'Number of airports: {df_airports_bronze.count()}')
print(f'Number of airlines: {df_airlines_bronze.count()}')

Number of flights: 5819079
Number of airports: 322
Number of airlines: 14


## BRONZE - PIPELINE - Column name standardization: change all column names to lowercase

In [13]:
def to_lowercase_columns(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, c.lower())
    return df

df_flights_bronze = to_lowercase_columns(df_flights_bronze)
df_airports_bronze = to_lowercase_columns(df_airports_bronze)
df_airlines_bronze = to_lowercase_columns(df_airlines_bronze)

## BRONZE - PIPELINE - add technical column

In [14]:
df_flights_bronze = df_flights_bronze.withColumn('load_date', F.current_timestamp())
df_flights_bronze = df_flights_bronze.withColumn('source_file', F.lit(ENV_file_name_fligths))

df_airports_bronze = df_airports_bronze.withColumn('load_date', F.current_timestamp())
df_airports_bronze = df_airports_bronze.withColumn('source_file', F.lit(ENV_file_name_airports))

df_airlines_bronze = df_airlines_bronze.withColumn('load_date', F.current_timestamp())
df_airlines_bronze = df_airlines_bronze.withColumn('source_file', F.lit(ENV_file_name_airlines))

## Save data as parquet file in bronze layer

In [15]:
'''
df_flights_bronze.write.mode("overwrite").parquet("bronze/flights")
df_airports_bronze.write.mode("overwrite").parquet("bronze/airports")
df_airlines_bronze.write.mode("overwrite").parquet("bronze/airlines")
'''

'\ndf_flights_bronze.write.mode("overwrite").parquet("bronze/flights")\ndf_airports_bronze.write.mode("overwrite").parquet("bronze/airports")\ndf_airlines_bronze.write.mode("overwrite").parquet("bronze/airlines")\n'

# Data Validation Tests
Validation functions are implemented as reusable pipeline components, while exploratory checks are performed only during development.

##  BRONZE - ANALYSIS - check data quality
flights are from a few year so consider: 'year','month','day' and maybe 'scheduled_time'  
check:  
'flight_number' and later add columns "airline", 'origin_airport','destination_airport' and check if 'tail_number' also need to be added

In [16]:
df_flights_bronze.groupBy('year','month','day','flight_number').count().filter('count > 1').limit(30).toPandas()

,year,month,day,flight_number,count
0,2015,1,1,674,3
1,2015,1,1,1872,3
2,2015,1,1,1926,4
3,2015,1,1,240,7
4,2015,1,1,364,7
5,2015,1,1,994,7
6,2015,1,1,1134,4
7,2015,1,1,4972,2
8,2015,1,1,5540,2
9,2015,1,1,3343,2


In [17]:
df_flights_bronze.groupBy('year','month','day','airline','flight_number','origin_airport').count().filter('count > 1').limit(30).toPandas()

,year,month,day,airline,flight_number,origin_airport,count
0,2015,8,29,AA,803,STT,2


In [18]:
df_flights_bronze.groupBy('year','month','day','airline','flight_number','origin_airport','destination_airport').count().filter('count > 1').limit(30).toPandas()

,year,month,day,airline,flight_number,origin_airport,destination_airport,count


In [19]:
df_flights_bronze.filter((F.col('year')=='2015') & (F.col('month')==1) & (F.col('day')==1) & (F.col('flight_number')== 1301)).limit(30).toPandas()

,year,month,day,day_of_week,airline,flight_number,tail_number,origin_airport,destination_airport,scheduled_departure,...,diverted,cancelled,cancellation_reason,air_system_delay,security_delay,airline_delay,late_aircraft_delay,weather_delay,load_date,source_file
0,2015,1,1,4,AA,1301,N487AA,MSP,DFW,820,...,0,1,B,NaN,NaN,NaN,NaN,NaN,2026-07-09 10:10:54.621037,flights.csv
1,2015,1,1,4,F9,1301,N923FR,ATL,IAD,1420,...,0,0,None,NaN,NaN,NaN,NaN,NaN,2026-07-09 10:10:54.621037,flights.csv


In [20]:
## airport uniqueness
df_airports_bronze.groupBy('iata_code').count().filter('count > 1').limit(30).toPandas()

,iata_code,count


In [21]:
df_airlines_bronze.groupBy('iata_code').count().filter('count > 1').limit(30).toPandas()

,iata_code,count


## BRONZE - PIPELINE - write function for checking uniqueness

In [22]:
def test_uniqueness(df, cols):

    duplicates = (
        df.groupBy(cols)
          .count()
          .filter(F.col("count") > 1)
    )

    failed_records = duplicates.count()

    return {
        "layer": "Bronze",
        "test": "Uniqueness",
        "status": "PASS" if failed_records == 0 else "FAIL",
        "failed_records": failed_records,
        "details": f"Business key: {', '.join(cols)}",
        "sample": duplicates.limit(5).toPandas() if failed_records else None
    }

## BRONZE - PIPELINE - (optional) if it is required to fail the pipeline when there is something wrong with data
below code will terminate the process

In [23]:
## check function
result = test_uniqueness(df_airports_bronze, ['iata_code'])

if result["status"] == "FAIL":
    raise Exception(result)

'''
# there is not uniqueness to this would crash pipeline
result = test_uniqueness(df_flights_bronze, ['year','month','day','flight_number']) 
if result["status"] == "FAIL":
    raise Exception(result)
'''


'\n# there is not uniqueness to this would crash pipeline\nresult = test_uniqueness(df_flights_bronze, [\'year\',\'month\',\'day\',\'flight_number\']) \nif result["status"] == "FAIL":\n    raise Exception(result)\n'

## BRONZE - ANALYSIS - check if not cancelled has delay time

In [24]:
df_flights_bronze.filter((F.col('cancelled')==1) & (F.col('arrival_delay').isNotNull())).limit(30).toPandas()

,year,month,day,day_of_week,airline,flight_number,tail_number,origin_airport,destination_airport,scheduled_departure,...,diverted,cancelled,cancellation_reason,air_system_delay,security_delay,airline_delay,late_aircraft_delay,weather_delay,load_date,source_file


In [25]:
df_flights_bronze.filter((F.col('cancelled')==1) & (F.col('arrival_delay').isNotNull())).limit(30).toPandas()

,year,month,day,day_of_week,airline,flight_number,tail_number,origin_airport,destination_airport,scheduled_departure,...,diverted,cancelled,cancellation_reason,air_system_delay,security_delay,airline_delay,late_aircraft_delay,weather_delay,load_date,source_file


## BRONZE - PIPELINE - write a function for this check if not cancelled has delay time

In [26]:
def test_cancelled_flights_without_delay_time(df):

    invalid_rows = df.filter(
        (F.col("cancelled") == 1)
        & F.col("arrival_delay").isNotNull()
    )

    failed_records = invalid_rows.count()

    return {
        "layer": "Bronze",
        "test": "Cancelled flights with arrival delay",
        "status": "PASS" if failed_records == 0 else "FAIL",
        "failed_records": failed_records,
        "details": "Cancelled flights should not contain arrival delay.",
        "sample": invalid_rows.limit(5).toPandas() if failed_records else None
    }

## BRONZE - ANALYSIS - Check if delay times can have negative values

In [27]:
df_flights_bronze.filter(F.col('arrival_delay')<0).limit(30).toPandas()

,year,month,day,day_of_week,airline,flight_number,tail_number,origin_airport,destination_airport,scheduled_departure,...,diverted,cancelled,cancellation_reason,air_system_delay,security_delay,airline_delay,late_aircraft_delay,weather_delay,load_date,source_file
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,0,0,None,NaN,NaN,NaN,NaN,NaN,2026-07-09 10:11:11.441585,flights.csv
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,0,0,None,NaN,NaN,NaN,NaN,NaN,2026-07-09 10:11:11.441585,flights.csv
2,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,0,0,None,NaN,NaN,NaN,NaN,NaN,2026-07-09 10:11:11.441585,flights.csv
3,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,0,0,None,NaN,NaN,NaN,NaN,NaN,2026-07-09 10:11:11.441585,flights.csv
4,2015,1,1,4,NK,612,N635NK,LAS,MSP,25,...,0,0,None,NaN,NaN,NaN,NaN,NaN,2026-07-09 10:11:11.441585,flights.csv
5,2015,1,1,4,US,2013,N584UW,LAX,CLT,30,...,0,0,None,NaN,NaN,NaN,NaN,NaN,2026-07-09 10:11:11.441585,flights.csv
6,2015,1,1,4,AA,1112,N3LAAA,SFO,DFW,30,...,0,0,None,NaN,NaN,NaN,NaN,NaN,2026-07-09 10:11:11.441585,flights.csv
7,2015,1,1,4,DL,1173,N826DN,LAS,ATL,30,...,0,0,None,NaN,NaN,NaN,NaN,NaN,2026-07-09 10:11:11.441585,flights.csv
8,2015,1,1,4,DL,2336,N958DN,DEN,ATL,30,...,0,0,None,NaN,NaN,NaN,NaN,NaN,2026-07-09 10:11:11.441585,flights.csv
9,2015,1,1,4,AA,1674,N853AA,LAS,MIA,35,...,0,0,None,NaN,NaN,NaN,NaN,NaN,2026-07-09 10:11:11.441585,flights.csv


## BRONZE - ANALYSIS - Count nulls in columns
Count total number and percentage of nulls

In [28]:
df_flights_bronze.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_flights_bronze.columns]).limit(30).toPandas()

,year,month,day,day_of_week,airline,flight_number,tail_number,origin_airport,destination_airport,scheduled_departure,...,diverted,cancelled,cancellation_reason,air_system_delay,security_delay,airline_delay,late_aircraft_delay,weather_delay,load_date,source_file
0,0,0,0,0,0,0,14721,0,0,0,...,0,0,5729195,4755640,4755640,4755640,4755640,4755640,0,0


## BRONZE - ANALYSIS - Count nulls in columns
Show vertically (in column, not row) to make result more readable

In [29]:
results = []
total_cnt = df_flights_bronze.count()
for c in df_flights_bronze.columns:
    nulls_in_cols = df_flights_bronze.filter(F.col(c).isNull()).count()
    results.append((c, nulls_in_cols, round((nulls_in_cols/total_cnt)*100,2)))

df_result = spark.createDataFrame(results, ["column_name", "null_count", "nulls_percentage"])
df_result.limit(30).toPandas()

,column_name,null_count,nulls_percentage
0,year,0,0.00
1,month,0,0.00
2,day,0,0.00
3,day_of_week,0,0.00
4,airline,0,0.00
5,flight_number,0,0.00
6,tail_number,14721,0.25
7,origin_airport,0,0.00
8,destination_airport,0,0.00
9,scheduled_departure,0,0.00


In [30]:
df_airports_bronze.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_airports_bronze.columns]).limit(30).toPandas()

,iata_code,airport,city,state,country,latitude,longitude,load_date,source_file
0,0,0,0,0,0,3,3,0,0


In [31]:
df_airlines_bronze.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_airlines_bronze.columns]).limit(30).toPandas()

,iata_code,airline,load_date,source_file
0,0,0,0,0


## BRONZE - ANALYSIS - Check the range in "distance","air_time","arrival_delay" columns in flights

In [32]:
df_flights_bronze.describe(["distance","air_time","arrival_delay"]).limit(30).toPandas()

,summary,distance,air_time,arrival_delay
0,count,5819079,5714008,5714008
1,mean,822.3564947305235,113.51162809012519,4.407057357987598
2,stddev,607.7842873170487,72.23082162028517,39.27129709388608
3,min,21,7,-87
4,max,4983,690,1971


## BRONZE - ANALYSIS - check values for time HHMM (missing lead zeros, 2400)

In [33]:
time_cols = ["scheduled_time","scheduled_departure","departure_time","scheduled_arrival","arrival_time"]

df_min = df_flights_bronze.agg(*[F.min(c).alias(f"{c}_min") for c in time_cols]).toPandas().T.reset_index()
df_min.columns = ["column", "min_value"]
df_max = df_flights_bronze.agg(*[F.max(c).alias(f"{c}_max") for c in time_cols]).toPandas().T.reset_index()
df_max.columns = ["column", "max_value"]

display(df_min)
display(df_max)

,column,min_value
0,scheduled_time_min,18
1,scheduled_departure_min,1
2,departure_time_min,1
3,scheduled_arrival_min,1
4,arrival_time_min,1


,column,max_value
0,scheduled_time_max,718
1,scheduled_departure_max,2359
2,departure_time_max,2400
3,scheduled_arrival_max,2400
4,arrival_time_max,2400


In [34]:
#df_flights_bronze.filter((F.col("scheduled_arrival")==2400) & (F.col("arrival_delay")>0)).display()
df_flights_bronze.filter((F.col("scheduled_arrival")==2400) & (F.col("arrival_delay")>0)).toPandas().T

,0
year,2015
month,3
day,14
day_of_week,6
airline,F9
flight_number,2014
tail_number,N923FR
origin_airport,TPA
destination_airport,PHL
scheduled_departure,2135


In [35]:
df_flights_bronze.filter(F.length(F.col("scheduled_time").cast("string"))<4).limit(5).toPandas()

,year,month,day,day_of_week,airline,flight_number,tail_number,origin_airport,destination_airport,scheduled_departure,...,diverted,cancelled,cancellation_reason,air_system_delay,security_delay,airline_delay,late_aircraft_delay,weather_delay,load_date,source_file
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,0,0,None,NaN,NaN,NaN,NaN,NaN,2026-07-09 10:13:07.308050,flights.csv
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,0,0,None,NaN,NaN,NaN,NaN,NaN,2026-07-09 10:13:07.308050,flights.csv
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,0,0,None,NaN,NaN,NaN,NaN,NaN,2026-07-09 10:13:07.308050,flights.csv
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,0,0,None,NaN,NaN,NaN,NaN,NaN,2026-07-09 10:13:07.308050,flights.csv
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,0,0,None,NaN,NaN,NaN,NaN,NaN,2026-07-09 10:13:07.308050,flights.csv


## BRONZE - ANALYSIS - check nulls in time columns

In [36]:
print(df_flights_bronze.filter(F.col("scheduled_departure").isNull()).count())
print(df_flights_bronze.filter(F.col("scheduled_arrival").isNull()).count())
print(df_flights_bronze.filter(F.col("departure_time").isNull()).count())
print(df_flights_bronze.filter(F.col("arrival_time").isNull()).count())

0


0


86153


92513


# Data Quality Report

In [37]:
quality_results = [
    # this one should fail
    test_uniqueness(df_flights_bronze, ['year','month','day','flight_number']),
    # this one should pass
    test_cancelled_flights_without_delay_time(df_flights_bronze),

]

quality_report = spark.createDataFrame(
    [
        {
            k: v
            for k, v in result.items()
            if k != "sample"
        }
        for result in quality_results
    ]
)

quality_report.toPandas()

,details,failed_records,layer,status,test
0,"Business key: year, month, day, flight_number",1394194,Bronze,FAIL,Uniqueness
1,Cancelled flights should not contain arrival d...,0,Bronze,PASS,Cancelled flights with arrival delay


# SILVER
Silver layer applies business transformations while preserving data granularity.

Main objectives:
- standardize timestamps
- implement business rules if existed
- clarify column names
- prepare hash record fordimensions 

## SILVER - PIPELINE - FLIGHTS DATA - Load data

In [38]:
excluded_columns = ["load_date", "source_file"]
df_flights_silver = df_flights_bronze.drop(*excluded_columns)

## SILVER - PIPELINE - FLIGHTS DATA - TIMESTAMP

### Handling time format
Challenges:
- 5 -> 00:05
- 55 -> 00:55
- 2400 -> 00:00 next day
- arrival may occur next day

### Introduce for columns:
- scheduled_departure
- departure_time
- wheels_off
- wheels_on
- scheduled_arrival
- arrival_time



## SILVER - PIPELINE - FLIGHTS DATA - change column name

In [39]:
rename_cols_map_flights = {
    'airline': 'airline_code'
}
df_flights_silver = df_flights_silver.withColumnsRenamed(rename_cols_map_flights)

## SILVER - PIPELINE - FLIGHTS DATA - function for date

In [40]:
def create_date_column(year_col, month_col, day_col):
    date_col = F.to_date(F.concat_ws("-", year_col, month_col, day_col), 'yyyy-M-d')
    return date_col

## SILVER - PIPELINE - FLIGHTS DATA - functions for timestamp

In [41]:
def create_timestamp_column(date_col, time_hhmm_col):
    
    # add leading zeros eg. 5 -> 0005, 830 -> 0830
    hhmm = F.lpad(F.col(time_hhmm_col).cast("string"), 4, "0")

    # fix 2400: change to 0000
    fixed_hhmm = F.when(hhmm == "2400", F.lit("0000")).otherwise(hhmm)

    # build timestamp
    # ts = F.to_timestamp(F.concat_ws(" ", date_col, fixed_hhmm), "yyyy-M-d HHmm")
    ts = F.when(hhmm.isNotNull(), F.to_timestamp(F.concat_ws(" ", date_col, fixed_hhmm), "yyyy-M-d HHmm"))
    
    # if 2400, add one day
    ts_corrected = F.when(hhmm == "2400",ts + F.expr("INTERVAL 1 DAY")).otherwise(ts)

    return ts_corrected

In [42]:
def add_flight_timestamps(df):
    return (
        df
        .withColumn("scheduled_departure_timestamp", create_timestamp_column("flight_date", "scheduled_departure"))
        .withColumn("scheduled_arrival_timestamp", create_timestamp_column("flight_date", "scheduled_arrival"))
        .withColumn("departure_timestamp", create_timestamp_column("flight_date", "departure_time"))
        .withColumn("arrival_timestamp", create_timestamp_column("flight_date", "arrival_time"))
    )

## SILVER - PIPELINE - FLIGHTS DATA - create timestamps

In [43]:
df_flights_silver_ts = df_flights_silver.withColumn("flight_date", create_date_column("year", "month", "day"))
df_flights_silver_ts = add_flight_timestamps(df_flights_silver_ts)
df_flights_silver_ts.limit(10).toPandas()

,year,month,day,day_of_week,airline_code,flight_number,tail_number,origin_airport,destination_airport,scheduled_departure,...,air_system_delay,security_delay,airline_delay,late_aircraft_delay,weather_delay,flight_date,scheduled_departure_timestamp,scheduled_arrival_timestamp,departure_timestamp,arrival_timestamp
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,NaN,NaN,NaN,NaN,NaN,2015-01-01,2015-01-01 00:05:00,2015-01-01 04:30:00,2015-01-01 23:54:00,2015-01-01 04:08:00
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,NaN,NaN,NaN,NaN,NaN,2015-01-01,2015-01-01 00:10:00,2015-01-01 07:50:00,2015-01-01 00:02:00,2015-01-01 07:41:00
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,NaN,NaN,NaN,NaN,NaN,2015-01-01,2015-01-01 00:20:00,2015-01-01 08:06:00,2015-01-01 00:18:00,2015-01-01 08:11:00
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,NaN,NaN,NaN,NaN,NaN,2015-01-01,2015-01-01 00:20:00,2015-01-01 08:05:00,2015-01-01 00:15:00,2015-01-01 07:56:00
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,NaN,NaN,NaN,NaN,NaN,2015-01-01,2015-01-01 00:25:00,2015-01-01 03:20:00,2015-01-01 00:24:00,2015-01-01 02:59:00
5,2015,1,1,4,DL,806,N3730B,SFO,MSP,25,...,NaN,NaN,NaN,NaN,NaN,2015-01-01,2015-01-01 00:25:00,2015-01-01 06:02:00,2015-01-01 00:20:00,2015-01-01 06:10:00
6,2015,1,1,4,NK,612,N635NK,LAS,MSP,25,...,NaN,NaN,NaN,NaN,NaN,2015-01-01,2015-01-01 00:25:00,2015-01-01 05:26:00,2015-01-01 00:19:00,2015-01-01 05:09:00
7,2015,1,1,4,US,2013,N584UW,LAX,CLT,30,...,NaN,NaN,NaN,NaN,NaN,2015-01-01,2015-01-01 00:30:00,2015-01-01 08:03:00,2015-01-01 00:44:00,2015-01-01 07:53:00
8,2015,1,1,4,AA,1112,N3LAAA,SFO,DFW,30,...,NaN,NaN,NaN,NaN,NaN,2015-01-01,2015-01-01 00:30:00,2015-01-01 05:45:00,2015-01-01 00:19:00,2015-01-01 05:32:00
9,2015,1,1,4,DL,1173,N826DN,LAS,ATL,30,...,NaN,NaN,NaN,NaN,NaN,2015-01-01,2015-01-01 00:30:00,2015-01-01 07:11:00,2015-01-01 00:33:00,2015-01-01 06:56:00


## SILVER - PIPELINE - hash fuction for scd2

### Row Hash Strategy
Each attribute is encoded as:
`column_name=value`

In [44]:
def build_row_hash(cols):
    record_hash = F.sha2(F.concat_ws("||",*[F.concat(F.lit(c), F.lit("="), F.coalesce(F.col(c).cast("string"), F.lit("NULL"))) for c in cols]), 256)
    return record_hash

## SILVER - PIPELINE - AIRPORTS - Load data

In [45]:
excluded_columns = ["load_date", "source_file"]
df_airports_silver = df_airports_bronze.drop(*excluded_columns)

## SILVER - PIPELINE - AIRPORTS -Change columns names
 - airports table iata_code to airport_code

In [46]:
rename_cols_map_airports = {
    'iata_code': 'airport_code'
}
df_airports_silver = df_airports_silver.withColumnsRenamed(rename_cols_map_airports)

## SILVER - PIPELINE - AIRPORTS - Add record hash for scd2

In [47]:
# exclude identifier columns
exclude_key = ["airport_code"]
cols_to_hash = [c for c in df_airports_silver.columns if c not in exclude_key]
df_airports_silver = df_airports_silver.withColumn("row_hash", build_row_hash(cols_to_hash))

## SILVER - PIPELINE - AIRLINES - Load data

In [48]:
excluded_columns = ["load_date", "source_file"]
df_airlines_silver = df_airlines_bronze.drop(*excluded_columns)

## SILVER - PIPELINE - AIRLINES - Change columns names
 - airlines table iata_code to airline_code

In [49]:
rename_cols_map_airlines = {
    'iata_code': 'airline_code'
}
df_airlines_silver = df_airlines_silver.withColumnsRenamed(rename_cols_map_airlines)
    

## SILVER - PIPELINE - AIRLINES - Add record hash for scd2

In [50]:
exclude_columns = ["airline_code"]
cols_to_hash = [c for c in df_airlines_silver.columns if c not in exclude_key]
df_airlines_silver = df_airlines_silver.withColumn("row_hash", build_row_hash(cols_to_hash))

# Gold
Provides a dimensional model optimized for analytical workloads

Main objects:
- dimensions
- fact table
- feature engineering

In [51]:
'''
df_silver = spark.read.parquet("silver/flights")
df_airports = spark.read.parquet("bronze/airports")
df_airlines = spark.read.parquet("bronze/airlines")
'''

'\ndf_silver = spark.read.parquet("silver/flights")\ndf_airports = spark.read.parquet("bronze/airports")\ndf_airlines = spark.read.parquet("bronze/airlines")\n'

## GOLD - PIPELINE - dim_time 
## IMPORTANT: executed once during warehouse initialization

In [52]:
START_DATE = "2010-01-01"
END_DATE = "2035-12-31"

# -----------------------------
# Generate dates
# -----------------------------

dim_time = spark.sql(f"""
    SELECT explode(
        sequence(
            to_date('{START_DATE}'),
            to_date('{END_DATE}'),
            interval 1 day
        )
    ) AS date
""")

# -----------------------------
# Create dimension
# -----------------------------

dim_time = dim_time.select(
    F.date_format("date", "yyyyMMdd").cast("int").alias("date_id"),
    F.col("date"),

    F.year("date").alias("year"),
    F.quarter("date").alias("quarter"),

    F.month("date").alias("month"),
    F.date_format("date", "MMMM").alias("month_name"),
    F.date_format("date", "MMM").alias("month_short"),

    F.weekofyear("date").alias("week"),

    F.dayofmonth("date").alias("day"),
    F.dayofweek("date").alias("day_of_week"),
    F.dayofyear("date").alias("day_of_year"),

    F.date_format("date", "EEEE").alias("day_name"),

    F.date_format("date", "yyyy-MM").alias("year_month"),
    F.concat(F.year("date"), F.lit("-W"), F.lpad(F.weekofyear("date"), 2, "0")).alias("year_week"),

    F.trunc("date", "month").alias("first_day_of_month"),
    F.last_day("date").alias("last_day_of_month"),

    (F.col("date") == F.last_day("date")).alias("is_month_end"),

    F.dayofweek("date").isin(1, 7).alias("is_weekend")
)
'''
if seasoning is important this could be also added:
F.when(F.month("date").isin(12, 1, 2), "Winter")
    .when(F.month("date").isin(3, 4, 5), "Spring")
    .when(F.month("date").isin(6, 7, 8), "Summer")
    .otherwise("Autumn")
    .alias("season")
'''
# -----------------------------
# Holidays calendar depends on country
# Below the example of adding
# -----------------------------
holidays = [
    ("20240101", "New Year"),
    ("20241225", "Christmas")
]

df_holidays = (spark.createDataFrame(holidays, ["date_id", "holiday_name"]))

dim_time = (
    dim_time
    .join(df_holidays, "date_id", "left")
    .withColumn("is_holiday", F.col("holiday_name").isNotNull())
)

dim_time.limit(10).toPandas()

,date_id,date,year,quarter,month,month_name,month_short,week,day,day_of_week,day_of_year,day_name,year_month,year_week,first_day_of_month,last_day_of_month,is_month_end,is_weekend,holiday_name,is_holiday
0,20100101,2010-01-01,2010,1,1,January,Jan,53,1,6,1,Friday,2010-01,2010-W53,2010-01-01,2010-01-31,False,False,None,False
1,20100103,2010-01-03,2010,1,1,January,Jan,53,3,1,3,Sunday,2010-01,2010-W53,2010-01-01,2010-01-31,False,True,None,False
2,20100106,2010-01-06,2010,1,1,January,Jan,1,6,4,6,Wednesday,2010-01,2010-W01,2010-01-01,2010-01-31,False,False,None,False
3,20100108,2010-01-08,2010,1,1,January,Jan,1,8,6,8,Friday,2010-01,2010-W01,2010-01-01,2010-01-31,False,False,None,False
4,20100104,2010-01-04,2010,1,1,January,Jan,1,4,2,4,Monday,2010-01,2010-W01,2010-01-01,2010-01-31,False,False,None,False
5,20100109,2010-01-09,2010,1,1,January,Jan,1,9,7,9,Saturday,2010-01,2010-W01,2010-01-01,2010-01-31,False,True,None,False
6,20100110,2010-01-10,2010,1,1,January,Jan,1,10,1,10,Sunday,2010-01,2010-W01,2010-01-01,2010-01-31,False,True,None,False
7,20100107,2010-01-07,2010,1,1,January,Jan,1,7,5,7,Thursday,2010-01,2010-W01,2010-01-01,2010-01-31,False,False,None,False
8,20100102,2010-01-02,2010,1,1,January,Jan,53,2,7,2,Saturday,2010-01,2010-W53,2010-01-01,2010-01-31,False,True,None,False
9,20100105,2010-01-05,2010,1,1,January,Jan,1,5,3,5,Tuesday,2010-01,2010-W01,2010-01-01,2010-01-31,False,False,None,False


## GOLD - DIM_AIRPORTS
### Key Design Decisions

**Business Key**  
- iata_code

**Surrogate Key**  
- airport_id (generated using row_number + max existing ID) 

**Change Detection**  
- SHA256 row hash using column-aware encoding:  
  column_name=value

**Versioning**  
- valid_from
- valid_to
- is_current



## GOLD - PIPELINE - DIM_AIRPORTS - load data

In [53]:
# make safe copy, not shallow copy
df_airports_gold = df_airports_silver.select(*df_airports_silver.columns)

## GOLD - DIM_AIRPORTS - initial load

In [54]:
# SCD2 simulation; global ordering acceptable despite Spark warning about missing partition (intentional single-partition design)
window = Window.orderBy("airport_code")

initial_dim = df_airports_gold.withColumn("airport_id", F.row_number().over(window))

dim_airports = (initial_dim
    #.withColumn("valid_from", F.current_timestamp()) # depends on approach
    .withColumn("valid_from", F.lit('1900-01-01').cast("timestamp"))
    .withColumn("valid_to", F.lit('3000-12-31').cast("timestamp"))
    .withColumn("is_current", F.lit(True))
)

# change the order of column to put at the begining id, hash, valid_from, valid_to, is_current
# technical columns are first because in case of dimension extention they will not be in the middle of table
dim_airports = dim_airports.select(
    "airport_id",
    "row_hash",
    "valid_from",
    "valid_to",
    "is_current",
    "airport_code",
    "airport",
    "city",
    "state",
    "country",
    "latitude",
    "longitude"
)

## GOLD - DIM_AIRPORTS - add "-1" and "-999" records

- **-1 (UNKNOWN MEMBER)**  
  Ensures referential integrity in dimensional models and allows facts to remain joinable even when dimension data is missing.

- **-999 (QUARANTINED RECORD)**  
  Indicates data that failed validation rules and should be excluded from analytical aggregations unless explicitly required for data quality monitoring.

In [55]:
# the order of column is important
unknown_airport = spark.createDataFrame(
    [
        Row(
            airport_id=-1,
            row_hash=None,
            valid_from=None,
            valid_to=None,
            is_current=True,
            airport_code="UNKNOWN",
            airport="Unknown Airport",
            city="Unknown",
            state="Unknown",
            country="Unknown",
            latitude=None,
            longitude=None
        )
    ],
    schema=dim_airports.schema
)

quarantined_airport = spark.createDataFrame(
    [
        Row(
            airport_id=-999,
            row_hash=None,
            valid_from=None,
            valid_to=None,
            is_current=True,
            airport_code="QUARANTINED",
            airport="Missing Airport",
            city="Unknown",
            state="Unknown",
            country="Unknown",
            latitude=None,
            longitude=None
        )
    ],
    schema=dim_airports.schema
)
#dim_airports = unknown_airport.unionByName(quarantined_airport).unionByName(dim_airports)
if (dim_airports.filter(F.col("airport_id") == -1).count() == 0) & (dim_airports.filter(F.col("airport_id") == -999).count() == 0):
    dim_airports = unknown_airport.unionByName(quarantined_airport).unionByName(dim_airports)
else:
    print("Records already existed")

In [56]:
'''
dim_airports.limit(10).toPandas()
doesn't work because converting to pandas changes timestamp columns to datetime64[ns]
which max is Timestamp('2262-04-11 23:47:16.854775807')
and valid_to is 3000-01-01
'''
(
    dim_airports.select(
        [F.col(c).cast("string").alias(c)
         if c in ["valid_to"] else F.col(c)
         for c in dim_airports.columns])
    .limit(10).toPandas()
)

,airport_id,row_hash,valid_from,valid_to,is_current,airport_code,airport,city,state,country,latitude,longitude
0,-1,None,NaT,None,True,UNKNOWN,Unknown Airport,Unknown,Unknown,Unknown,NaN,NaN
1,-999,None,NaT,None,True,QUARANTINED,Missing Airport,Unknown,Unknown,Unknown,NaN,NaN
2,1,3ee4e61e6bb79f4087692689e042b98c27ac51e0e84764...,1900-01-01,3000-12-31 00:00:00,True,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
3,2,a10b7984c849486313cb38bb2b9ca8316129016971f9cf...,1900-01-01,3000-12-31 00:00:00,True,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
4,3,0d3fea02e1c8485a39c799bc97ef24a4df9c03a36285a7...,1900-01-01,3000-12-31 00:00:00,True,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919
5,4,67452d0ee858e3883654ead2e9c1e438a458685351b206...,1900-01-01,3000-12-31 00:00:00,True,ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183
6,5,8ac8c40bfcf8c19d55438d7b52344709e321a10f13359f...,1900-01-01,3000-12-31 00:00:00,True,ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447
7,6,73384b854259be92e1d3be06b417e0eb88a45ce309c379...,1900-01-01,3000-12-31 00:00:00,True,ACK,Nantucket Memorial Airport,Nantucket,MA,USA,41.25305,-70.06018
8,7,ba7c53e898311de7b1de834e44daf72dd8c1e8ceee5779...,1900-01-01,3000-12-31 00:00:00,True,ACT,Waco Regional Airport,Waco,TX,USA,31.61129,-97.23052
9,8,af4c2ce700dc65806ccc810071590bd061e955a461bf84...,1900-01-01,3000-12-31 00:00:00,True,ACV,Arcata Airport,Arcata/Eureka,CA,USA,40.97812,-124.10862


## GOLD - PIPELINE - DIM_AIPORTS - SCD2 table load
Considering possibility that records could be remover from source

In [57]:
#########################
# # simulate incoming new data
# ABE - changed record
# ABQ - deleted record (removed from source)

df_new_batch = (df_airports_silver
                .drop("row_hash")
                .filter(F.col("airport_code") != "ABQ")
                .withColumn("airport", 
                            F.when(F.col("airport_code") == "ABE", F.lit("Lehigh Valley International Airport New"))
                            .otherwise(F.col("airport")))
                )
# build hash
exclude_columns = ["airport_code"]
cols_to_hash = [c for c in df_new_batch.columns if c not in exclude_columns]
df_new_batch = df_new_batch.withColumn("row_hash",build_row_hash(cols_to_hash))
#########################

# current dimension
current_dim = dim_airports.filter(F.col("is_current") & (~F.col("airport_id").isin(-1, -999)))

# detect SCD2 changes
records_with_status = (
    df_new_batch.alias("new")
    .join(current_dim.alias("old"), "airport_code", "full")
    .withColumn("airport_code_key", F.coalesce(F.col("new.airport_code"), F.col("old.airport_code")))
    .withColumn(
        "record_status",
        F.when(F.col("old.airport_code").isNull(), "NEW")
         .when(F.col("new.airport_code").isNull(), "DELETED")
         .when(F.col("new.row_hash") != F.col("old.row_hash"), "CHANGED")
         .otherwise("UNCHANGED")
    )
)

# records requiring expiration
keys_to_expire = (
    records_with_status
    .filter(F.col("record_status").isin("CHANGED", "DELETED"))
    .select(F.col("airport_code_key").alias("airport_code"))
    .distinct()
)

# close old versions
df_expired = (
    dim_airports
    .filter(F.col("is_current"))
    .join(keys_to_expire, "airport_code")
    .withColumn("valid_to", F.current_timestamp())
    .withColumn("is_current", F.lit(False))
)

# keep unchanged records and historical records
df_unchanged = dim_airports.join(keys_to_expire, "airport_code", "left_anti")


# prepare new versions of existed records and records for new business keys
records_to_insert = (
    records_with_status
    .filter(F.col("record_status").isin("NEW", "CHANGED"))
    .select("new.*")
)


# generate surrogate keys
max_id = dim_airports.agg(F.max("airport_id").alias("max_id")).first()["max_id"] or 0

df_insert = (
    records_to_insert
    .withColumn("airport_id", F.row_number().over(Window.orderBy("airport_code")) + F.lit(max_id))
    .withColumn("valid_from", F.current_timestamp())
    .withColumn("valid_to", F.lit("3000-12-31").cast("timestamp"))
    .withColumn("is_current", F.lit(True))
)

# rebuild dimension
dim_airports_updated = (
    df_unchanged
    .unionByName(df_expired)
    .unionByName(df_insert)
)

# final validation
print("------------------------------------")
print("Final dimension counts:")
print("------------------------------------")
display(dim_airports_updated.groupBy("is_current").count().limit(30).toPandas())

print("------------------------------------")
print("Records by status:")
print("------------------------------------")
display(records_with_status.groupBy("record_status").count().limit(30).toPandas())

display(
    dim_airports.orderBy("airport_code", "valid_from").select(
        [F.col(c).cast("string").alias(c)
         if c in ["valid_to"] else F.col(c)
         for c in dim_airports.columns])
    .limit(10).toPandas()
)


------------------------------------
Final dimension counts:
------------------------------------


,is_current,count
0,True,323
1,False,2


------------------------------------
Records by status:
------------------------------------


,record_status,count
0,CHANGED,1
1,DELETED,1
2,UNCHANGED,320


,airport_id,row_hash,valid_from,valid_to,is_current,airport_code,airport,city,state,country,latitude,longitude
0,1,3ee4e61e6bb79f4087692689e042b98c27ac51e0e84764...,1900-01-01,3000-12-31 00:00:00,True,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
1,2,a10b7984c849486313cb38bb2b9ca8316129016971f9cf...,1900-01-01,3000-12-31 00:00:00,True,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
2,3,0d3fea02e1c8485a39c799bc97ef24a4df9c03a36285a7...,1900-01-01,3000-12-31 00:00:00,True,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919
3,4,67452d0ee858e3883654ead2e9c1e438a458685351b206...,1900-01-01,3000-12-31 00:00:00,True,ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183
4,5,8ac8c40bfcf8c19d55438d7b52344709e321a10f13359f...,1900-01-01,3000-12-31 00:00:00,True,ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447
5,6,73384b854259be92e1d3be06b417e0eb88a45ce309c379...,1900-01-01,3000-12-31 00:00:00,True,ACK,Nantucket Memorial Airport,Nantucket,MA,USA,41.25305,-70.06018
6,7,ba7c53e898311de7b1de834e44daf72dd8c1e8ceee5779...,1900-01-01,3000-12-31 00:00:00,True,ACT,Waco Regional Airport,Waco,TX,USA,31.61129,-97.23052
7,8,af4c2ce700dc65806ccc810071590bd061e955a461bf84...,1900-01-01,3000-12-31 00:00:00,True,ACV,Arcata Airport,Arcata/Eureka,CA,USA,40.97812,-124.10862
8,9,dc0c550177071b0ecce3c106c3dea10cd038c83727e300...,1900-01-01,3000-12-31 00:00:00,True,ACY,Atlantic City International Airport,Atlantic City,NJ,USA,39.45758,-74.57717
9,10,226e3bc2e73bf51e29b5599ca1a0e18f7cfc27e58f3039...,1900-01-01,3000-12-31 00:00:00,True,ADK,Adak Airport,Adak,AK,USA,51.87796,-176.64603


## GOLD - DIM_AIRLINES
### Key Design Decisions

**Business Key**  
- iata_code

**Surrogate Key**  
- airline_id (generated using row_number + max existing ID) 

**Change Detection**  
- SHA256 row hash using column-aware encoding:  
  column_name=value

**Versioning**  
- valid_from
- valid_to
- is_current



## GOLD - PIPELINE - DIM_AIRLINES - load data

In [58]:
# make safe copy, not shallow copy
df_airlines_gold = df_airlines_silver.select(*df_airlines_silver.columns)

## GOLD - DIM_AIRLINES - initial load

In [59]:
# SCD2 simulation; global ordering acceptable despite Spark warning about missing partition (intentional single-partition design)
window = Window.orderBy("airline_code")

initial_dim_airlines = df_airlines_gold.withColumn("airline_id", F.row_number().over(window))

dim_airlines = (initial_dim_airlines
    .withColumn("valid_from", F.lit('1900-01-01').cast("timestamp"))
    .withColumn("valid_to", F.lit('3000-12-31').cast("timestamp"))
    .withColumn("is_current", F.lit(True))
)

# change the order of column to put at the begining id, hash, valid_from, valid_to, is_current
# technical columns are first because in case of dimension extention they will not be in the middle of table
dim_airlines = dim_airlines.select(
    "airline_id",
    "row_hash",
    "valid_from",
    "valid_to",
    "is_current",
    "airline_code",
    "airline"
)

## GOLD - DIM_AIRPORTS - add "-1" and "-999" records

- **-1 (UNKNOWN MEMBER)**  
  Ensures referential integrity in dimensional models and allows facts to remain joinable even when dimension data is missing.

- **-999 (QUARANTINED RECORD)**  
  Indicates data that failed validation rules and should be excluded from analytical aggregations unless explicitly required for data quality monitoring.

In [60]:
'''
dim_airlines.limit(10).toPandas()
doesn't work because converting to pandas changes timestamp columns to datetime64[ns]
which max is Timestamp('2262-04-11 23:47:16.854775807')
and valid_to is 3000-01-01
'''
(
    dim_airlines.select(
        [F.col(c).cast("string").alias(c)
         if c in ["valid_to"] else F.col(c)
         for c in dim_airlines.columns])
     .limit(10).toPandas()
)

,airline_id,row_hash,valid_from,valid_to,is_current,airline_code,airline
0,1,eee0ab33558b15946f2d7d290daccd30d55f1a512d4ee7...,1900-01-01,3000-12-31 00:00:00,True,AA,American Airlines Inc.
1,2,986e84cc84d3baa4cad8dca004814c561e57325c0d2585...,1900-01-01,3000-12-31 00:00:00,True,AS,Alaska Airlines Inc.
2,3,aae76d7f6fa1ee0f6ac20ab37c482761b1d7f9d5ab1da9...,1900-01-01,3000-12-31 00:00:00,True,B6,JetBlue Airways
3,4,f6413bec953f3cea63a1c9ec7802b87b8ff40d6e202587...,1900-01-01,3000-12-31 00:00:00,True,DL,Delta Air Lines Inc.
4,5,9ed119f8799aefa6052405152eccbcffcf1c13b78accfa...,1900-01-01,3000-12-31 00:00:00,True,EV,Atlantic Southeast Airlines
5,6,a158d0c6a2499ba6a00fffc23ac4cd546a9e27f098adf3...,1900-01-01,3000-12-31 00:00:00,True,F9,Frontier Airlines Inc.
6,7,9cfcdc22d0007759800cfa7eeb9b91305d1ca9e2675ac9...,1900-01-01,3000-12-31 00:00:00,True,HA,Hawaiian Airlines Inc.
7,8,d23b78fa45176d8933c9e6d11f561ae68e26ad1bd606ab...,1900-01-01,3000-12-31 00:00:00,True,MQ,American Eagle Airlines Inc.
8,9,c0fdc74215336fc78032d19b9e78f46194eb0a21fe4c85...,1900-01-01,3000-12-31 00:00:00,True,NK,Spirit Air Lines
9,10,01f391ba08622b2156f152f06237ef95e3a2c905d75818...,1900-01-01,3000-12-31 00:00:00,True,OO,Skywest Airlines Inc.


In [61]:
# the order of column is important
unknown_row_airlines = spark.createDataFrame([
    Row(
        airline_id=-1,
        row_hash="NA",
        valid_from=None,
        valid_to=None,
        is_current=True,
        airline_code="UNKNOWN",
        airline="Unknown Airline"
        )
    ],
    schema=dim_airlines.schema
)

quarantined_row_airlines = spark.createDataFrame([
    Row(
        airline_id=-999,
        row_hash="NA",
        valid_from=None,
        valid_to=None,
        is_current=True,
        airline_code="QUARANTINED",
        airline="Missing Airline"
        )
    ],
    schema=dim_airlines.schema
)

if (dim_airlines.filter(F.col("airline_id") == -1).count() == 0) & (dim_airlines.filter(F.col("airline_id") == -999).count() == 0):
    dim_airlines = unknown_row_airlines.unionByName(quarantined_row_airlines).unionByName(dim_airlines)
else:
    print("Records already existed")

## GOLD - PIPELINE - DIM_LINES - SCD2 table load
Considering possibility that records could be remover from source

In [62]:
# without changes simulation
df_new_batch = df_airlines_silver
# current dimension
current_dim = dim_airlines.filter(F.col("is_current") & (~F.col("airline_id").isin(-1, -999)))

# detect SCD2 changes
records_with_status = (
    df_new_batch.alias("new")
    .join(current_dim.alias("old"), "airline_code", "full")
    .withColumn("airline_code_key", F.coalesce(F.col("new.airline_code"), F.col("old.airline_code")))
    .withColumn(
        "record_status",
        F.when(F.col("old.airline_code").isNull(), "NEW")
         .when(F.col("new.airline_code").isNull(), "DELETED")
         .when(F.col("new.row_hash") != F.col("old.row_hash"), "CHANGED")
         .otherwise("UNCHANGED")
    )
)

# records requiring expiration
keys_to_expire = (
    records_with_status
    .filter(F.col("record_status").isin("CHANGED", "DELETED"))
    .select(F.col("airline_code_key").alias("airline_code"))
    .distinct()
)

# close old versions
df_expired = (
    dim_airlines
    .filter(F.col("is_current"))
    .join(keys_to_expire, "airline_code")
    .withColumn("valid_to", F.current_timestamp())
    .withColumn("is_current", F.lit(False))
)

# keep unchanged records and historical records
df_unchanged = dim_airlines.join(keys_to_expire, "airline_code", "left_anti")


# prepare new versions of existed records and records for new business keys
records_to_insert = (
    records_with_status
    .filter(F.col("record_status").isin("NEW", "CHANGED"))
    .select("new.*")
)


# generate surrogate keys
max_id = dim_airlines.agg(F.max("airline_id").alias("max_id")).first()["max_id"] or 0

df_insert = (
    records_to_insert
    .withColumn("airline_id", F.row_number().over(Window.orderBy("airline_code")) + F.lit(max_id))
    .withColumn("valid_from", F.current_timestamp())
    .withColumn("valid_to", F.lit("3000-12-31").cast("timestamp"))
    .withColumn("is_current", F.lit(True))
)

# rebuild dimension
dim_airlines_updated = (
    df_unchanged
    .unionByName(df_expired)
    .unionByName(df_insert)
)

# final validation
print("------------------------------------")
print("Final dimension counts:")
print("------------------------------------")
display(dim_airlines_updated.groupBy("is_current").count().limit(5).toPandas())

print("------------------------------------")
print("Records by status:")
print("------------------------------------")
display(records_with_status.groupBy("record_status").count().limit(5).toPandas())

display(
    dim_airlines.orderBy("airline_code", "valid_from").select(
        [F.col(c).cast("string").alias(c)
         if c in ["valid_to"] else F.col(c)
         for c in dim_airlines.columns])
     .limit(10).toPandas()
)

------------------------------------
Final dimension counts:
------------------------------------


,is_current,count
0,True,16


------------------------------------
Records by status:
------------------------------------


,record_status,count
0,UNCHANGED,14


,airline_id,row_hash,valid_from,valid_to,is_current,airline_code,airline
0,1,eee0ab33558b15946f2d7d290daccd30d55f1a512d4ee7...,1900-01-01,3000-12-31 00:00:00,True,AA,American Airlines Inc.
1,2,986e84cc84d3baa4cad8dca004814c561e57325c0d2585...,1900-01-01,3000-12-31 00:00:00,True,AS,Alaska Airlines Inc.
2,3,aae76d7f6fa1ee0f6ac20ab37c482761b1d7f9d5ab1da9...,1900-01-01,3000-12-31 00:00:00,True,B6,JetBlue Airways
3,4,f6413bec953f3cea63a1c9ec7802b87b8ff40d6e202587...,1900-01-01,3000-12-31 00:00:00,True,DL,Delta Air Lines Inc.
4,5,9ed119f8799aefa6052405152eccbcffcf1c13b78accfa...,1900-01-01,3000-12-31 00:00:00,True,EV,Atlantic Southeast Airlines
5,6,a158d0c6a2499ba6a00fffc23ac4cd546a9e27f098adf3...,1900-01-01,3000-12-31 00:00:00,True,F9,Frontier Airlines Inc.
6,7,9cfcdc22d0007759800cfa7eeb9b91305d1ca9e2675ac9...,1900-01-01,3000-12-31 00:00:00,True,HA,Hawaiian Airlines Inc.
7,8,d23b78fa45176d8933c9e6d11f561ae68e26ad1bd606ab...,1900-01-01,3000-12-31 00:00:00,True,MQ,American Eagle Airlines Inc.
8,9,c0fdc74215336fc78032d19b9e78f46194eb0a21fe4c85...,1900-01-01,3000-12-31 00:00:00,True,NK,Spirit Air Lines
9,10,01f391ba08622b2156f152f06237ef95e3a2c905d75818...,1900-01-01,3000-12-31 00:00:00,True,OO,Skywest Airlines Inc.


## GOLD - FACT_FLIGHTS
**Fact grain**

One record = one scheduled flight  

**Primary business key:**

- flight_date
- flight_number
- airline_id
- origin_airport_id
- destination_airport_id

## GOLD - PIPELINE - FACT_FLIGHTS - Load data

In [63]:
excluded_cols = ['year','month','day','scheduled_departure', 'departure_time', 'scheduled_arrival', 'arrival_time']
df_flights_gold = df_flights_silver_ts.drop(*excluded_cols)

##  GOLD - PIPELINE - FACT_FLIGHTS - Business rules implementation (feature engineering)

Implemented features:

- Flight length categories (based on IATA https://en.wikipedia.org/wiki/Flight_length )
- departure delay categories
- arrival delay categories

In [64]:
df_flights_gold = df_flights_gold.withColumn("flights_length_category",
                                             F.when(F.col("air_time") < 300, "short")
                                             .when(F.col("air_time") < 3600, "medium")
                                             .otherwise("long"))

df_flights_gold = df_flights_gold.withColumn("departure_delay_category",
                                             F.when(F.col("departure_delay") < 0, "Early")
                                             .when(F.col("departure_delay") < 15, "On Time")
                                             .when(F.col("departure_delay") < 30, "Small Delay")
                                             .when(F.col("departure_delay") < 60, "Medium Delay")
                                             .when(F.col("departure_delay") < 120, "Long Delay")
                                             .otherwise("Critical Delay")
                                             )

df_flights_gold = df_flights_gold.withColumn("arrival_delay_category",
                                             F.when(F.col("arrival_delay") < 0, "Early")
                                             .when(F.col("arrival_delay") < 15, "On Time")
                                             .when(F.col("arrival_delay") < 30, "Small Delay")
                                             .when(F.col("arrival_delay") < 60, "Medium Delay")
                                             .when(F.col("arrival_delay") < 120, "Long Delay")
                                             .otherwise("Critical Delay")
                                             )

##  GOLD - PIPELINE - FACT_FLIGHTS - mark missing values before lookups
if business keys are missing that is a quality data issue

In [65]:
df_flights_gold = df_flights_gold.fillna({
     "origin_airport": "QUARANTINED",
    "destination_airport": "QUARANTINED",
    "airline_code": "QUARANTINED"        
})

## GOLD - PIPELINE - FACT_FLIGHTS - lookups
Add key - ( including dim_airports, dim_airlines, dim_time)

In [66]:
# -------------------
# Lookup dimensions
# -------------------

dim_airports_lkp = dim_airports.select(
    "airport_code",
    "airport_id",
    "valid_from",
    "valid_to"
)

dim_airlines_lkp = dim_airlines.select(
    "airline_code",
    "airline_id",
    "valid_from",
    "valid_to"
)

# -------------------
# Prepare fact
# -------------------

df_flights_gold = (
    df_flights_gold
    .withColumn("date_id", F.date_format("flight_date", "yyyyMMdd").cast("int"))
)

# -------------------
# Origin airport lookup
# -------------------

df_flights_gold = (
    df_flights_gold.alias("fct")
    .join(
        dim_airports_lkp.alias("d_ap"),
        (F.col("fct.origin_airport") == F.col("d_ap.airport_code")) &
        (F.col("fct.departure_timestamp") >= F.col("d_ap.valid_from")) &
        (F.col("fct.departure_timestamp") < F.col("d_ap.valid_to")),
        "left"
    )
    .select(
        "fct.*",
        F.coalesce(F.col("d_ap.airport_id"), F.lit(-1)).alias("origin_airport_id")
    )
)

# -------------------
# Destination airport lookup
# -------------------

df_flights_gold = (
    df_flights_gold.alias("fct")
    .join(
        dim_airports_lkp.alias("d_ap"),
        (F.col("fct.destination_airport") == F.col("d_ap.airport_code")) &
        (F.col("fct.departure_timestamp") >= F.col("d_ap.valid_from")) &
        (F.col("fct.departure_timestamp") < F.col("d_ap.valid_to")),
        "left"
    )
    .select(
        "fct.*",
        F.coalesce(F.col("d_ap.airport_id"), F.lit(-1)).alias("destination_airport_id")
    )
)

# -------------------
# Airline lookup
# -------------------

df_flights_gold = (
    df_flights_gold.alias("fct")
    .join(
        dim_airlines_lkp.alias("d_al"),
        (F.col("fct.airline_code") == F.col("d_al.airline_code")) &
        (F.col("fct.departure_timestamp") >= F.col("d_al.valid_from")) &
        (F.col("fct.departure_timestamp") < F.col("d_al.valid_to")),
        "left"
    )
    .select(
        "fct.*",
        F.coalesce(F.col("d_al.airline_id"), F.lit(-1)).alias("airline_id")
    )
)

##  GOLD - PIPELINE - FACT_FLIGHTS - add technical columns

In [67]:
df_flights_gold = df_flights_gold.withColumn("run_id", F.date_format(F.current_timestamp(), "yyyyMMddHHmmss"))

## GOLD - PIPELINE - FCT_FLIGHT - final column list

In [68]:
col_list = [
 'run_id',
 'date_id',
 'origin_airport_id',
 'destination_airport_id',
 'airline_id',
 'flight_number',
 'airline_code',
 'tail_number',
 'origin_airport',
 'destination_airport',
 'departure_delay',
 'taxi_out',
 'wheels_off',
 'scheduled_time',
 'elapsed_time',
 'air_time',
 'distance',
 'wheels_on',
 'taxi_in',
 'arrival_delay',
 'diverted',
 'cancelled',
 'cancellation_reason',
 'air_system_delay',
 'security_delay',
 'airline_delay',
 'late_aircraft_delay',
 'weather_delay',
 'flight_date',
 'scheduled_departure_timestamp',
 'scheduled_arrival_timestamp',
 'departure_timestamp',
 'arrival_timestamp',
 'flights_length_category',
 'departure_delay_category',
 'arrival_delay_category',]

df_flights_gold = df_flights_gold.select(*col_list)
df_flights_gold.limit(5).toPandas()

,run_id,date_id,origin_airport_id,destination_airport_id,airline_id,flight_number,airline_code,tail_number,origin_airport,destination_airport,...,late_aircraft_delay,weather_delay,flight_date,scheduled_departure_timestamp,scheduled_arrival_timestamp,departure_timestamp,arrival_timestamp,flights_length_category,departure_delay_category,arrival_delay_category
0,20260709101343,20150617,21,279,11,1900,UA,N12216,ATL,SFO,...,8.0,0.0,2015-06-17,2015-06-17 07:20:00,2015-06-17 09:31:00,2015-06-17 07:44:00,2015-06-17 09:52:00,short,Small Delay,Small Delay
1,20260709101343,20150617,152,183,11,1906,UA,N37274,IAH,LGA,...,NaN,NaN,2015-06-17,2015-06-17 07:20:00,2015-06-17 11:59:00,2015-06-17 07:19:00,2015-06-17 11:40:00,short,Early,Early
2,20260709101343,20150327,21,229,10,5566,OO,N124SY,ATL,ORD,...,NaN,NaN,2015-03-27,2015-03-27 13:21:00,2015-03-27 14:29:00,2015-03-27 13:18:00,2015-03-27 14:15:00,short,Early,Early
3,20260709101343,20150327,21,183,8,3031,MQ,N530MQ,ATL,LGA,...,0.0,0.0,2015-03-27,2015-03-27 13:21:00,2015-03-27 15:30:00,2015-03-27 14:57:00,2015-03-27 17:01:00,short,Long Delay,Long Delay
4,20260709101343,20150101,18,278,2,98,AS,N407AS,ANC,SEA,...,NaN,NaN,2015-01-01,2015-01-01 00:05:00,2015-01-01 04:30:00,2015-01-01 23:54:00,2015-01-01 04:08:00,short,Early,Early


In [69]:
'''
df_flights_gold.write \
    .mode("append") \
    .parquet("/data/gold/fct_flights")
'''

'\ndf_flights_gold.write     .mode("append")     .parquet("/data/gold/fct_flights")\n'

# Analytics mart data
Without business keys and technical columns

In [70]:
# remove bussines keys ans technical columns
excluded_cols = ['origin_airport','destination_airport','airline_code', 'run_id']
fact_flights_mart = df_flights_gold.select(*df_flights_gold.columns) 

# Analytics

## Question1: Which airlines are the most punctual/reliable?

In [71]:
(fact_flights_mart.alias("fct")
.join(dim_airlines.filter(F.col("is_current")).alias("d_al"), "airline_id")
.groupBy("d_al.airline")
.agg(
    F.count("*").alias("total_flights"),
    F.round(F.avg("departure_delay"), 2).alias("avg_departure_delay"),
    F.round(F.avg("arrival_delay"), 2).alias("avg_arrival_delay"),
    F.round(F.avg(F.when(F.col("arrival_delay_category").isin(["On Time", "Early"]) ,1)
                  .otherwise(0)) * 100,2).alias("on_time_rate_pct")
)
.orderBy("on_time_rate_pct", ascending=False)
).toPandas()


,airline,total_flights,avg_departure_delay,avg_arrival_delay,on_time_rate_pct
0,Hawaiian Airlines Inc.,76119,0.49,2.02,88.58
1,Alaska Airlines Inc.,171910,1.79,-0.98,86.72
2,Delta Air Lines Inc.,872177,7.37,0.19,86.25
3,American Airlines Inc.,715598,8.90,3.45,81.42
4,Skywest Airlines Inc.,579086,7.80,5.85,80.99
5,US Airways Inc.,194825,6.14,3.71,80.93
6,Southwest Airlines Co.,1246129,10.58,4.37,80.71
7,Virgin America,61385,9.02,4.74,80.59
8,Atlantic Southeast Airlines,557294,8.72,6.59,79.95
9,United Air Lines Inc.,509534,14.44,5.43,79.10


## Business Insights
- Hawaiian and Alaska consistently achieve the best on-time performance
- Frontier and Spirit show the weakest operational performance

## Question 2: Which airports generate the largest delays?

In [72]:
(fact_flights_mart.alias("fct")
.join(dim_airports.alias("d_ap").filter(F.col("is_current")), F.col("fct.origin_airport_id") == F.col("d_ap.airport_id"))
.groupBy("d_ap.airport", "d_ap.city", "d_ap.state")
.agg(
    F.count("*").alias("flights"),
    F.round(F.avg("fct.departure_delay"),2).alias("avg_departure_delay"),
    F.max("fct.departure_delay").alias("max_departure_delay")
)
.orderBy(F.desc("avg_departure_delay"))
).toPandas()

,airport,city,state,flights,avg_departure_delay,max_departure_delay
0,Wilmington Airport,Wilmington,DE,97,29.39,272
1,Martha's Vineyard Airport,Marthas Vineyard,MA,205,25.91,420
2,Barnstable Municipal Airport,Hyannis,MA,82,23.18,315
3,St. Cloud Regional Airport,St Cloud,MN,78,18.69,320
4,Southwest Oregon Regional Airport (North Bend ...,North Bend,OR,265,17.78,308
...,...,...,...,...,...,...
318,Merle K. (Mudhole) Smith Airport,Cordova,AK,653,-3.26,328
319,Valdez Airport,Vernal,UT,200,-3.74,626
320,Elko Regional Airport,Elko,NV,517,-3.77,566
321,Canyonlands Field,Moab,UT,205,-6.06,78


## Business Insights
- Airports with quite small number of flights show the highest average departure delays
- Major airport also face significant delays

## Question 3: How does flight punctuality change throughout the year?

In [73]:
(
    fact_flights_mart
    .join(dim_time.alias("d_tm"), "date_id")
    .groupBy("d_tm.month", "d_tm.month_name")
    .agg(
        F.count("*").alias("total_flights"),
        F.round(F.avg("departure_delay"), 2).alias("avg_departure_delay"),
        F.round(F.avg("arrival_delay"), 2).alias("avg_arrival_delay"),
        F.round(F.avg(F.when(F.col("departure_delay_category") == "On Time", 1).otherwise(0)) * 100, 2).alias("on_time_rate_pct"))
    .orderBy("month")
).toPandas()

,month,month_name,total_flights,avg_departure_delay,avg_arrival_delay,on_time_rate_pct
0,1,January,469968,9.76,5.81,23.39
1,2,February,429191,11.89,8.32,24.36
2,3,March,504312,9.66,4.92,25.10
3,4,April,485151,7.72,3.16,23.58
4,5,May,496993,9.45,4.49,23.47
5,6,June,503897,13.99,9.60,25.18
6,7,July,520718,11.39,6.43,24.88
7,8,August,510536,9.93,4.61,24.33
8,9,September,464946,4.82,-0.77,21.65
9,10,October,486165,4.98,-0.78,23.55


## Business Insights
- Operational performance declines during peak travel seasons (summer/winter holidays and christmas)
- September and October achieve the lowest average departure delays

# Project Summary
Implemented components:

✔ Bronze layer  
✔ Silver layer  
✔ Gold layer  
✔ SCD2  
✔ Data Quality  
✔ Star Schema  
✔ Feature Engineering  
✔ Analytics

## good practices:
- predefined schema
- no inferSchema
- no Python UDF
- surrogate keys
- SCD Type 2
- medallion architecture
- star schema
- technical columns
- data quality validation
- feature engineering
- lookup dimensions
